# Notebook do Navega Aí

Trabalho de IA Generativa - Professor: Dr. Jean de Lima

## Dependências

In [ ]:
%pip install --upgrade streamlit\
    langchain\
    langchain-community\
    langchain-huggingface\
    langchain-chroma\
    watchdog\
    pypdf\
    chromadb\
    langchain-groq\
    groq\
    sentence-transformers\
    python-dotenv\
    protobuf<4\
    opentelemetry-exporter-otlp<1.22

In [ ]:
import uuid
import os
import re
import shutil
import time

from dotenv import load_dotenv
from pathlib import Path

import streamlit as st

from langchain_chroma import Chroma

from langchain_community.document_loaders import (TextLoader, PyPDFLoader)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (RunnablePassthrough, RunnableParallel)
from langchain_core.messages import AIMessage, HumanMessage

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter



## Configuração do RAG

In [ ]:
load_dotenv()

# Modo offline para Hugging Face
# No deploy, preferimos permitir download online (melhora a inicialização).
os.environ.pop("TRANSFORMERS_OFFLINE", None)
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

BASE_DIR = Path(__file__).parent.parent

DOCS_DIR = BASE_DIR / "data" / "docs"
CHROMA_DIR = BASE_DIR / "data" / "chroma"
COLLECTION_NAME = "navega_ai_ufrn"
HF_TOKEN = os.getenv("HF_TOKEN", None)
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

AVAILABLE_MODELS = {
    "Llama 3.3 70B": {
        "model_id": "llama-3.3-70b-versatile",
        "temperature": 0.2,
        "description": "Modelo principal da Meta hospedado pela Groq. Melhor qualidade."
    },

    "Llama 3.1 8B": {
        "model_id": "llama-3.1-8b-instant",
        "temperature": 0.2,
        "description": "Modelo menor e extremamente rápido."
    },
}

TOP_K = 8
FETCH_K = 10

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

## Carregar arquivos

In [ ]:
def load_documents(folder):
    documents = []

    for file in Path(folder).rglob("*"):
        if not file.is_file():
            continue

        suffix = file.suffix.lower()

        try:
            if suffix in (".md", ".txt", ".markdown"):
                loader = TextLoader(str(file), encoding="utf-8")
                docs = loader.load()

            elif suffix == ".pdf":
                loader = PyPDFLoader(str(file))
                docs = loader.load()

            else:
                continue

            for doc in docs:
                if doc.page_content:
                    doc.page_content = (str(doc.page_content).replace("\x00", ""))

                doc.metadata["arquivo"] = file.name
                doc.metadata["categoria"] = file.parent.name

            documents.extend(docs)
            print(f"Carregado: {file.name} ({len(docs)} páginas/chunks)")

        except Exception as e:
            print( f"Erro ao carregar {file.name}: {e}")

    print(f"Total de documentos carregados: {len(documents)}")
    return documents

## Dividindo em chunks

In [ ]:
def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=[
            "\n# ",
            "\n## ",
            "\n### ",
            "\n\n",
            "\n",
            ". ",
            " ",
            ""
        ]
    )

    return splitter.split_documents(documents)

## Criando embeddings

In [ ]:
def load_embeddings():
    return HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={
            "device": "cpu",
        },
        encode_kwargs={
            "normalize_embeddings": True
        }
    )

## Criando banco vetorial

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', text)
    text = text.replace('\uf02d', '')
    text = text.replace('\u200b', '')
    text = text.replace('\u200c', '')
    text = text.replace('\u200d', '')
    text = text.replace('\ufeff', '')
    return text.strip()


def get_existing_vector_store(embeddings):
    chroma_dir = CHROMA_DIR
    chroma_db_file = chroma_dir / "chroma.sqlite3"

    if not chroma_db_file.exists():
        return None

    try:
        print(f"Carregando vector store existente em: {chroma_dir}")
        return Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=str(chroma_dir)
        )
    except Exception as e:
        msg = str(e).lower()
        if ("pickle" in msg) or ("eof" in msg) or ("hnsw" in msg) or ("deserial" in msg):
            print(f"Falha ao carregar vector store (provável corrupção/incompatibilidade). Recriando... Erro: {e}")
            if chroma_dir.exists():
                shutil.rmtree(chroma_dir)
            return None
        raise



def create_vector_store(documents, embeddings, force_rebuild=False):
    chroma_dir = CHROMA_DIR
    chroma_db_file = chroma_dir / "chroma.sqlite3"

    if not force_rebuild and chroma_db_file.exists():
        print("Vector store existente encontrado. Carregando...")
        return Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=str(chroma_dir)
        )

    if force_rebuild and chroma_dir.exists():
        print("Forçando rebuild. Apagando vector store existente...")
        shutil.rmtree(chroma_dir)

    print(f"Total de documentos recebidos: {len(documents)}")

    textos_validos = []
    metadados_validos = []

    for i, doc in enumerate(documents):
        if not hasattr(doc, 'page_content'):
            continue

        content = doc.page_content

        if content is None:
            continue

        if not isinstance(content, str):
            try:
                content = str(content)
            except:
                continue

        content = clean_text(content)

        if not content:
            continue

        textos_validos.append(content)
        metadados_validos.append(doc.metadata)

    print(f"Total de documentos válidos: {len(textos_validos)}")

    if not textos_validos:
        raise ValueError("Nenhum documento com conteúdo válido para indexar.")

    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(chroma_dir)
    )

    BATCH_SIZE = 50
    total_adicionados = 0

    for i in range(0, len(textos_validos), BATCH_SIZE):
        batch_texts = textos_validos[i:i + BATCH_SIZE]
        batch_metadatas = metadados_validos[i:i + BATCH_SIZE] if metadados_validos else None
        batch_ids = [str(uuid.uuid4()) for _ in range(len(batch_texts))]

        try:
            vector_store.add_texts(
                texts=batch_texts,
                metadatas=batch_metadatas,
                ids=batch_ids
            )
            total_adicionados += len(batch_texts)

        except Exception as e:
            print(f"Erro no batch {i}: {e}. Tentando documento por documento...")
            for j, (text, metadata, doc_id) in enumerate(zip(batch_texts, batch_metadatas, batch_ids)):
                try:
                    text = clean_text(text)
                    if text:
                        vector_store.add_texts(
                            texts=[text],
                            metadatas=[metadata] if metadata else None,
                            ids=[doc_id]
                        )
                        total_adicionados += 1
                except Exception as e2:
                    print(f"Documento {i+j} com falha: {e2}")

    print(f"Total adicionado ao vector store: {total_adicionados}")

    return vector_store

## Implementando prompt para o RAG

In [ ]:
# Melhorar prompt depois, para poder usar um modelo menor.
# No projeto, usamos o modelo de 70B, mas a meta é usar um de 4B no máximo,
# pois é algo simples e não exige muita complexidade.

SYSTEM_PROMPT = """
Você é um assistente de IA da UFRN.
Responda perguntas usando somente o contexto recuperado.
Se a informação não existir no contexto, responda exatamente: Não encontrei a resposta nos documentos disponíveis.

Para perguntas de carga horária, ou horas mínimas no IMDtec e no módulo integrador, responda em formato de tabela com as colunas Curso Módulo | Carga horária mínima | Fonte.

Você pode realizar cálculos matemáticos simples (ex.: subtrações e somas) usando valores que estejam explicitamente presentes no contexto.

Para outras perguntas, responda de forma objetiva e direta.

Não copie o contexto integralmente.
Fuja de informações falsas e inventadas.
Sempre cite a fonte quando ela estiver no contexto.
"""

## Construindo a chain

In [ ]:
from langchain_groq import ChatGroq

# Bem, eu usei o groq aqui, mas a meta é usar o hugging face.
# Depois de entregar os arquivos, irei modificar isso com o tempo,
# pois penso em disponibilizar o código para ser usado pela UFRN.

def get_model_config(model_key):
    if model_key in AVAILABLE_MODELS:
        return AVAILABLE_MODELS[model_key]

    return list(AVAILABLE_MODELS.values())[0]

def format_docs(docs):
    formatted = []

    for i, doc in enumerate(docs, 1):
        print(f"\nDOC {i}")
        print(doc.metadata)

        print(doc.page_content)

        source = doc.metadata.get("arquivo", "Desconhecido")
        category = doc.metadata.get("categoria", "Geral")

        formatted.append(
            f"[Documento {i} - Fonte: {source} "
            f"(Categoria: {category})]\n"
            f"{doc.page_content}\n"
        )

    return "\n\n".join(formatted)


def create_chain(
    vector_store,
    model_key=None,
    debug_retriever=False,
):

    model_config = get_model_config(model_key)

    llm = ChatGroq(
        model=model_config["model_id"],
        api_key=os.getenv("GROQ_API_KEY"),
        temperature=model_config["temperature"],
    )

    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": TOP_K,
        },
    )

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", SYSTEM_PROMPT,),
            ("human", "Contexto:\n{context}\n\n" "Pergunta:\n{question}" ),
        ]
    )

    chain = (
        RunnableParallel(
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
        )
        | prompt
        | llm
        | StrOutputParser()
    )

    return chain

## Construindo a interface de consulta com Streamlit

### Construindo o chat

In [ ]:
def init_chat_history():
    if "messages" not in st.session_state:
        st.session_state.messages = [
            AIMessage(
                content="Olá, como posso lhe ajudar hoje? 😊"
            )
        ]

def add_message(role, content):
    if role == "user":
        st.session_state.messages.append(HumanMessage(content=content))
    else:
        st.session_state.messages.append(AIMessage(content=content))

def display_chat_history():
    for message in st.session_state.messages:
        role = "user" if isinstance(message, HumanMessage) else "assistant"
        with st.chat_message(role):
            st.markdown(message.content)

def display_streaming_response(response_text):
    with st.chat_message("assistant"):
        st.markdown(response_text)
    return response_text

def clear_chat_history():
    st.session_state.messages = []
    if "current_response" in st.session_state:
        del st.session_state.current_response

### Contruindo a barra lateral

In [ ]:
def render_sidebar():
    with st.sidebar:
        st.markdown(
            """
            **Assistente Virtual do IMDtec**
            
            Tire dúvidas sobre:
            - o IMDtec
            - Módulo Integrador
            - Grades Curriculares
            - Processos Burocráticos
            """
        )

        st.markdown("---")

        st.markdown("### Modelo de IA")
        model_keys = list(AVAILABLE_MODELS.keys())
        default_idx = 0  # Qwen é o padrão
                         # * Fiz esse comentário quando usei o hugging faces

        selected_model = st.selectbox(
            "Escolha o modelo:",
            options=model_keys,
            index=default_idx,
            format_func=lambda k: k.split(" (")[0],  # Mostra só o nome (ex: "Qwen2.5-1.5B")
                                                     # * Este também!
            help="Selecione o modelo de linguagem para responder às perguntas."
        )
        
        st.caption(
            AVAILABLE_MODELS[selected_model].get(
                "description",
                "Sem descrição."
            )
        )

        if "selected_model" not in st.session_state:
            st.session_state.selected_model = selected_model
        elif st.session_state.selected_model != selected_model:
            st.session_state.selected_model = selected_model
            st.session_state.model_changed = True
            st.rerun()

        st.markdown("---")

        if st.button("Limpar conversa", use_container_width=True):
            clear_chat_history()
            st.rerun()

        if st.button("Atualizar documentos", use_container_width=True):
            st.session_state.force_rebuild = True
            st.rerun()

        st.markdown("---")

        st.caption("Desenvolvido no IMDtec")

### Juntando tudo!

In [ ]:
st.set_page_config(
    page_title="Navega Aí!",
    page_icon="🚢",
    layout="centered"
)

init_chat_history()
render_sidebar()

if "debug_retriever" not in st.session_state:
    st.session_state.debug_retriever = False

if "selected_model" not in st.session_state:
    st.session_state.selected_model = list(AVAILABLE_MODELS.keys())[0]

current_model = st.session_state.selected_model
model_changed = st.session_state.pop("model_changed", False)

if "vector_store" not in st.session_state or st.session_state.get("force_rebuild"):
    with st.spinner("Carregando documentos e criando vector store..."):
        embeddings = load_embeddings()
        vector_store = get_existing_vector_store(embeddings)

        if vector_store is None or st.session_state.get("force_rebuild"):
            documents = load_documents(DOCS_DIR)
            chunks = split_documents(documents)
            vector_store = create_vector_store(
                chunks,
                embeddings,
                force_rebuild=st.session_state.get("force_rebuild", False)
            )

        st.session_state.vector_store = vector_store
        st.session_state.vector_store_ready = True
        st.session_state.force_rebuild = False

if "chain" not in st.session_state or model_changed or "chain_model" not in st.session_state or st.session_state.chain_model != current_model:
    model_name_display = current_model.split(" (")[0]
    with st.spinner(
        f"Carregando modelo **{model_name_display}**... "
        "Isso pode levar alguns minutos na primeira execução de cada modelo."
    ):
        
        if "chain" in st.session_state:
            del st.session_state.chain
            import gc
            gc.collect()

        chain = create_chain(
            st.session_state.vector_store,
            model_key=current_model,
            debug_retriever=st.session_state.get("debug_retriever", False),
        )
        st.session_state.chain = chain
        st.session_state.chain_ready = True
        st.session_state.chain_model = current_model

st.title("🚢 Navega Aí!")

display_chat_history()
question = st.chat_input("Digite sua pergunta sobre a UFRN...")

if question:
    add_message("user", question)
    with st.chat_message("assistant"):
        with st.spinner("Consultando documentos e gerando resposta..."):
            try:
                start = time.time()
                response = st.session_state.chain.invoke(question)
                elapsed = time.time() - start
                print(f"Tempo da consulta: {elapsed:.2f}s")
                st.markdown(response)
                add_message("assistant", response)
            except Exception as e:
                error_msg = f" **Erro ao gerar resposta:** {str(e)}"
                st.error(error_msg)
                add_message("assistant", error_msg)

    st.rerun()